# Voice cloning — what is actually brokenOne speaker, one recording session, both languages. Four arms that separatethree things every earlier run confounded: a noisy reference, a hardcross-lingual task, and whatever XTTS-v2 does badly on its own.| arm | reference | speaks | answers ||---|---|---|---|| `en -> en` | English | English | XTTS's best case. If this fails, nothing else matters || `hi -> hi` | Hindi | Hindi | Monolingual Hindi — the model's ceiling || `en -> hi` | English | Hindi | **Production** || `hi -> en` | Hindi | English | Does the loss travel one way or both |The number that makes the rest readable is none of those. It is the **realHindi recording scored against the anchor built from the real English one** —same human, two languages, no synthesis. Speaker embeddings shift acrosslanguages even for a real person, so that is the honest ceiling for `en -> hi`.Every earlier run scored cross-lingual synthesis against a same-languageceiling and charged the model for a gap the metric creates by itself.Reference audio ships in the repo under `fixtures/`, so **Part 1 needs noupload**. Part 3 is the real dubbing run and does need the bundle zip.

## 1. Check the GPUIf this prints nothing: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. Clone and install`colab/requirements.txt` pins `transformers>=4.57,<5` on purpose. `coqui-tts`imports `isin_mps_friendly`, which transformers 5.x removed, and Colabpreinstalls 5.x — so without the pin the import fails at model load. It alsopins no torch version at all, because Colab's preinstalled torch is built forits own CUDA and replacing it breaks more than it fixes.

In [ ]:
%cd /content!git clone -q -b test https://github.com/ayushk1233/indic-dub-pipeline.git 2>/dev/null || echo "already cloned"%cd /content/indic-dub-pipeline!git fetch --all --quiet && git checkout --quiet test && git pull --quiet!pip install -q -r colab/requirements.txtprint("\ninstalled — now restart the runtime before running anything else")

## 3. Restart the runtime — not optionalThe install downgraded `transformers`. Anything already imported in thissession still holds the old module objects, and the failure shows up muchlater as a confusing `ImportError` inside the model loader.**Runtime → Restart session**, then continue from cell 4. Do not re-run cell 2.

## 4. Verify the environment after the restart

In [ ]:
%cd /content/indic-dub-pipelineimport transformers, torchprint("transformers", transformers.__version__, "| torch", torch.__version__,      "| cuda", torch.cuda.is_available())from pathlib import Pathfor name in sorted(p.name for p in Path("fixtures").glob("*")):    size = Path("fixtures", name).stat().st_size    print(f"  {name:<28} {size/1e6:6.2f} MB")

## 5. The experimentLoads XTTS once, measures the calibration scale, then runs all four arms underboth greedy and sampled decoding. Roughly 6–10 minutes on a T4.Conditioning is held at the values Coqui shipped in XTTS-v2's own`config.json`. The earlier sweep of seven conditioning settings spanned 0.046on a scale now known to span about 0.80, so conditioning is not a variableworth moving here. Language is the variable under test.

In [ ]:
import importlib, colab.four_arm as four_armimportlib.reload(four_arm)rows = four_arm.main()

## 6. ListenReal recordings first, so you have the target in your ears, then each arm.Judge three things **separately** — they fail independently and have differentfixes:1. **Identity** — is it the same person as the real recording?2. **Humanness** — would you believe a person said it, ignoring who?3. **Accent** — Indian or American? The similarity metric cannot see this at all.

In [ ]:
four_arm.listen(rows)

## 7. Download the reportSend me `four_arm_report.txt` along with your answers to those three questions.

In [ ]:
from google.colab import filesfiles.download("/content/four_arm_report.txt")

---## 8. The real dubbing run — optional, run after Part 1Everything above is diagnosis on isolated sentences. This synthesizes theactual 14-segment bundle the local pipeline exported from `english.mov`.Upload `artifacts/english_clean/tts_bundle.zip`.

In [ ]:
import zipfilefrom pathlib import Pathfrom google.colab import filesuploaded = files.upload()BUNDLE = Path("/content/tts_bundle")BUNDLE.mkdir(exist_ok=True)with zipfile.ZipFile(next(iter(uploaded)), "r") as z:    z.extractall(BUNDLE)print(f"\n{len(list((BUNDLE / 'request').glob('*')))} request files")print(f"reference: {(BUNDLE / 'request' / 'reference.wav').stat().st_size/1e6:.2f} MB")

In [ ]:
from colab.xtts_worker import XTTSWorkerworker = XTTSWorker(BUNDLE)worker.load_bundle()worker.run()

QC report for the run — pace, speaker similarity and per-segment verdicts.

In [ ]:
import sysfor name in [n for n in list(sys.modules) if n.startswith("src.eval")]:    del sys.modules[name]from src.eval.harness import build_report, render_reportprint(render_report(build_report(job_dir=BUNDLE, bundle_dir=BUNDLE)))

In [ ]:
import shutilshutil.make_archive("/content/synthesis_output", "zip", BUNDLE / "output")files.download("/content/synthesis_output.zip")

---## What to send back1. `four_arm_report.txt`2. Identity / humanness / accent, in words, for each of the four arms3. If you ran Part 8: `synthesis_output.zip` and the QC tableThe decision rule is already fixed, so the numbers alone settle most of it:- `en -> en` high → the old reference was the whole problem- `en -> en` still low → XTTS cannot clone this voice; fine-tuning is on the table- `hi -> hi` high but `en -> hi` low → cross-lingual gap → voice conversion, not fine-tuning- `en -> hi` near its ceiling → identity is done; naturalness decides, and IndicF5 is next